In [1]:
# Import packages
import numpy as np
import pandas as pd
import hdbscan
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import datasets, linear_model, metrics
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
import statsmodels.api as sm
from functions_fcsuml import *
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



In [2]:
blobs, labels, centers = datasets.make_blobs(n_samples=90000, n_features=150, centers=20, random_state=42, return_centers=True)

targets = blobs[:, -1]
data = blobs[:, :-1]
print(data.shape, targets.shape)

(90000, 149) (90000,)


In [3]:
training_data, testing_data, training_targets, testing_targets = train_test_split(data, targets, test_size=0.2, random_state=42)

testing_data, validation_data, testing_targets, validation_targets = train_test_split(testing_data, testing_targets, test_size=0.5, random_state=42)

print('X1 shape: ', training_data.shape)
print('X2 shape: ', testing_data.shape)
print('X3 shape: ', validation_data.shape)
print('X4 shape: ', training_targets.shape)
print('X5 shape: ', testing_targets.shape)
print('X6 shape: ', validation_targets.shape)

# Normalizing the data
norm_data, normalization_variables = normalize(training_data)
norm_targets, max_targets, min_targets = normalize_feature(training_targets)

norm_testing_targets = (testing_targets-min_targets)/(max_targets-min_targets)
norm_validation_targets = (validation_targets-min_targets)/(max_targets-min_targets)
norm_validation_data = (validation_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
norm_testing_data = (testing_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
print(np.shape(norm_data))

X1 shape:  (72000, 149)
X2 shape:  (9000, 149)
X3 shape:  (9000, 149)
X4 shape:  (72000,)
X5 shape:  (9000,)
X6 shape:  (9000,)
(72000, 149)


# Raw data

In [6]:
X_norm_data = sm.add_constant(norm_data)

model_sm = sm.OLS(norm_targets, X_norm_data)
results_sm = model_sm.fit()

print(results_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.968
Model:                            OLS   Adj. R-squared:                  0.968
Method:                 Least Squares   F-statistic:                 1.473e+04
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        11:57:33   Log-Likelihood:             1.2265e+05
No. Observations:               72000   AIC:                        -2.450e+05
Df Residuals:                   71850   BIC:                        -2.436e+05
Df Model:                         149                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7575      0.024     31.221      0.0

# Embedded data

In [ ]:
number_of_clusters, optimal_clustering = clustering(norm_data, cluster_size=300, min_samples=2)



TypeError: tuple indices must be integers or slices, not tuple

In [25]:
print(number_of_clusters)
def create_clusterspan(sorted_clusters, number_of_clusters):
    clusterspan = np.zeros(number_of_clusters+2)

    for i in range(0, number_of_clusters+1):
        clusterspan[i+1] = clusterspan[i] + np.shape(sorted_clusters[np.where(sorted_clusters[:,0] == i)])[0]

    clusterspan[-1] = np.shape(sorted_clusters)[0]
    return clusterspan

19


In [26]:
sorted_clusters = optimal_clustering[optimal_clustering[:, 0].argsort()] # sorting the clustered data by cluster affiliation
number_of_clusters = int(np.max(sorted_clusters[:,0]))
clusterspan = create_clusterspan(sorted_clusters, number_of_clusters)
print(clusterspan)
average_silhouette, cluster_silhouettes = silhouette(sorted_clusters, number_of_clusters)
print('avg sil:', average_silhouette, '\n cluster silhouettes: \n', cluster_silhouettes, '\n argmax', np.argmax(cluster_silhouettes[:,1]))
show_clusters_in_latlon(norm_data, sorted_clusters, number_of_clusters, clusterspan,2, 1)

[    0.  4500.  9000. 13500. 18000. 22500. 27000. 31500. 36000. 40500.
 45000. 49500. 54000. 58500. 63000. 67500. 72000. 76500. 81000. 85500.
 90000.]


KeyboardInterrupt: 

In [12]:
def center_of_mass(number_of_clusters, clusters):
    """Takes a number of clusters and a matrix with points declaring where the first column references which cluster to which the point belongs and
    the rest the coordinates of the point."""
    cluster_centers = np.zeros((number_of_clusters, len(clusters[0])))
    balancing_vector = np.zeros_like(cluster_centers)
    clusters[:,0] += 1
    for i in range(len(clusters[:,None])):
        if clusters[i,0] > 0:
            cluster_centers[int(clusters[i,0])-1] += clusters[i]
            balancing_vector[int(clusters[i,0])-1] += 1 # Keeps track of how many points are in each cluster.
    return cluster_centers / balancing_vector

def center_of_mass(number_of_clusters, clusters):

    cluster_centers = np.zeros((number_of_clusters, np.shape(clusters)[1]))
    # print(cluster_centers.shape)
    # print(clusters[np.where(clusters[:,0] == 1)])
    # balancing_vector = np.zeros_like
    for i in range(1, number_of_clusters+1):
        cluster_x = clusters[np.where(clusters[:,0] == i)]
        # print(np.shape(cluster_x))
        cluster_centers[i-1,:] = np.sum(cluster_x, axis=0) / np.shape(cluster_x)[0]

    return cluster_centers


def vector_norm(vector):
    return np.sqrt(vector @ vector.T)

def create_unit_vector(vector):
    """Takes a vector and creates a unit vector in the same direction."""
    unit_vector = vector / vector_norm(vector)
    return unit_vector

def create_basis(centers_of_mass):
    """Takes the centers_of_mass array and outputs a transformation matrix."""
    basis = np.zeros_like(centers_of_mass[:, 1:])
    for i in range(len(centers_of_mass[:, None])):
        basis[i] = create_unit_vector(centers_of_mass[i, 1:])

    return basis

centers_of_mass = center_of_mass(number_of_clusters, sorted_clusters)
print(centers_of_mass)
transformation_matrix = create_basis(centers_of_mass)

"""
print(np.shape(transformation_matrix@norm_data.T))
print(centers_of_mass)
print(transformation_matrix)
print(transformation_matrix[1,:]@transformation_matrix[1,:].T)
print(transformation_matrix@norm_data.T)
"""

embedded_data = (transformation_matrix@norm_data.T).T

# norm_embedded_data = normalize(embedded_data)

# print(embedded_data)
print(np.shape(embedded_data))
# print(norm_embedded_data)

[[ 1.         -7.65617332  8.79907271 ... -4.16075778  7.94525941
  -9.70855199]
 [ 2.         -2.50644385  8.9986018  ...  2.20079925  0.02491979
  -8.98602609]
 [ 3.          8.15749514 -5.19675129 ...  5.41957527 -5.69818715
   2.4518349 ]
 ...
 [17.         -8.94960519  0.61842828 ...  9.71157549  5.07755818
  -2.4748088 ]
 [18.          0.37832751 -0.38480101 ... -7.44068935  6.51724812
   5.65035188]
 [19.         -6.61653308 -4.41041954 ... -2.32894823  0.84091097
   8.12673063]]
(72000, 19)


In [13]:
embedded_testing_data = (transformation_matrix@norm_testing_data.T).T
X_embedded_data = sm.add_constant(embedded_data)

model_sm = sm.OLS(norm_targets, X_embedded_data)
results_sm = model_sm.fit()

print(results_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.962
Model:                            OLS   Adj. R-squared:                  0.962
Method:                 Least Squares   F-statistic:                 9.717e+04
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:02:55   Log-Likelihood:             1.1657e+05
No. Observations:               72000   AIC:                        -2.331e+05
Df Residuals:                   71980   BIC:                        -2.329e+05
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5686      0.001    779.428      0.0

# PCA

In [ ]:
max_n_components = norm_data.shape[1]
r2_mat = np.zeros(max_n_components)

for n in range(max_n_components):
    pca = PCA(n_components=n)
    pca.fit(norm_data)

    pca_embed = pca.fit_transform(norm_data)
    # print(pca_embed.shape)
    # print(pca.explained_variance_ratio_)
    # print(pca.singular_values_)
    

    pca_embedded_data = sm.add_constant(pca_embed)

    pca_model_sm = sm.OLS(norm_targets, pca_embedded_data)
    pca_results_sm = pca_model_sm.fit()
    r2_mat[n] = pca_results_sm.rsquared_adj

plt.plot(r2_mat)


In [21]:
pca = PCA(n_components=20)
pca.fit(norm_data)

pca_embed = pca.fit_transform(norm_data)
print(pca_embed.shape)
# print(pca.explained_variance_ratio_)
# print(pca.singular_values_)


pca_embedded_data = sm.add_constant(pca_embed)

pca_model_sm = sm.OLS(norm_targets, pca_embedded_data)
pca_results_sm = pca_model_sm.fit()

print(pca_results_sm.summary())

(72000, 20)
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.968
Model:                            OLS   Adj. R-squared:                  0.968
Method:                 Least Squares   F-statistic:                 1.096e+05
Date:                Fri, 27 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:22:39   Log-Likelihood:             1.2255e+05
No. Observations:               72000   AIC:                        -2.451e+05
Df Residuals:                   71979   BIC:                        -2.449e+05
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4218      0.000   2565.